# 3 — Activities

**Theme:** labelling periods of a record so you can analyse them separately.

An *activity* is a named set of time periods — a task, a process step, a
background period. Once marked, activities drive the statistics in
[4 — Statistics and exposure](04-statistics-and-exposure.ipynb) and the shading
in [5 — Plotting](05-plotting.ipynb).

Activities are stored in absolute time, so the same definitions can be applied
to every instrument in a campaign.

In [1]:
import aerosoltools as at

smps = at.load_smps_file("../../tests/data/Sample_SMPS.txt")
smps.activities

/opt/hostedtoolcache/Python/3.11.15/x64/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['All data']

Every dataset starts with a single built-in activity, `"All data"`, covering the
whole record.

## Marking known periods

The usual case: you know when each task ran. Pass a dictionary of activity name
to a list of `(start, end)` pairs — an activity can have several occurrences.

In [2]:
activity_periods = {
    "Emission": [
        ("2018-02-27 10:18:00", "2018-02-27 10:31:00"),
        ("2018-02-27 10:35:00", "2018-02-27 10:48:00"),
        ("2018-02-27 10:52:00", "2018-02-27 11:30:00"),
        ("2018-02-27 12:39:00", "2018-02-27 12:48:00"),
    ],
    "Constant phase 1": [("2018-02-27 11:43:00", "2018-02-27 12:35:00")],
    "Constant phase 2": [("2018-02-27 12:55:00", "2018-02-27 13:45:00")],
    "Background": [("2018-02-27 13:48:00", "2018-02-27 14:39:00")],
}

smps.mark_activities(activity_periods)
smps.activities

['All data', 'Emission', 'Constant phase 1', 'Constant phase 2', 'Background']

`.activity_periods` gives the periods back, as absolute timestamps.

In [3]:
smps.activity_periods["Background"]

[('2018-02-27 13:48:00', '2018-02-27 14:39:00')]

## Marking by threshold

When you do not have a task log, `mark_threshold` labels every sample on one
side of a value.

In [4]:
cpc = at.load_cpc_file("../../tests/data/Sample_CPC_AIM.txt")
cpc.mark_threshold("Elevated", threshold=20000, threshold_direction="above")
cpc.activities

['All data', 'Elevated']

## Finding peaks

`peak_finder` marks samples that rise above a rolling baseline — useful for
picking out short emission events without knowing when they happened. It flags
a sample when the signal exceeds `baseline + ratio * rolling_std`, where the
baseline is a rolling median over `window` samples.

In [5]:
cpc.peak_finder(window=15, ratio=2.5, method="median")
cpc.activities

['All data', 'Elevated', 'Peak']

## Renaming

`rename_activity` keeps the periods and the mask, changing only the label.

In [6]:
cpc.rename_activity("Elevated", "Above 20k")
cpc.activities

['All data', 'Above 20k', 'Peak']

## Getting the data back out

`get_activity_data` returns only the rows inside an activity's periods — across
all of its occurrences.

In [7]:
emission = smps.get_activity_data("Emission")
background = smps.get_activity_data("Background")

print(f"Emission   : {len(emission)} samples")
print(f"Background : {len(background)} samples")
print(f"All data   : {len(smps.get_activity_data('All data'))} samples")

Emission   : 70 samples
Background : 48 samples
All data   : 243 samples


`get_activity_extra_data` does the same for the auxiliary channels in
`.extra_data`.

In [8]:
cpc.get_activity_extra_data("Above 20k").head()

,Sample Length,Averaging Interval,Title,Instrument ID,Instrument Errors,Mean,Min,Max,Std. Dev.,Comments
Datetime,,,,,,,,,,


Because the result is a plain DataFrame, ordinary pandas works from here.

In [9]:
emission["Total_conc"].describe()

count        70.000000
mean     579664.800100
std      157042.765229
min         228.462000
25%      504918.976500
50%      568887.091000
75%      719367.812000
max      798665.837000
Name: Total_conc, dtype: float64

That is the manual route. The next notebook does this across every activity at
once, with the statistics that matter for exposure assessment.

---

**Next:** [4 — Statistics and exposure](04-statistics-and-exposure.ipynb).